# Demo: Inverse Probability of Treatment Weighting (IPTW)

`IPTWWeighter` is the propensity-score-based counterpart to `MAICWeighter`/`EntropyBalanceWeighter`
(see the [MAIC / entropy balancing demo](maic_weighting.ipynb)): like those, it **reweights every
pool patient** rather than selecting a subset. Where entropy balancing solves directly for weights
that exactly balance chosen moments, IPTW instead:

1. fits a propensity model $p(X) = P(\mathrm{target} \mid X)$ that classifies pool vs. target
   patients on their covariates (`sklearn.linear_model.LogisticRegression` by default), then
2. weights each pool patient by the odds $p / (1 - p)$.

The target population keeps weight 1, so this is the **ATT estimand** with the target playing the
role of the fixed/reference ("treated") group -- the same convention `MAICWeighter` uses, and the
natural fit for this package's typical use case of reweighting a real-world data (RWD) pool to
represent a trial population.

Unlike entropy balancing, IPTW:
- **requires a patient-level target** (there's no pool-vs-target classification problem to fit
  against a published, aggregate-only Table 1);
- **does not guarantee exact balance** on any particular moment -- only asymptotically, and only
  if the propensity model is well specified;
- is nonetheless the most widely used and reported method for this kind of reweighting in the
  observational literature (Rosenbaum & Rubin, 1983; Austin, 2011).

In [ ]:
import logging

import matplotlib.pyplot as plt
import pandas as pd

from pybalance.sim import generate_toy_dataset
from pybalance.utils import MatchingData, MatchingHeaders
from pybalance.visualization import plot_categoric_features, plot_numeric_features
from pybalance.weighting import IPTWWeighter, MAICWeighter, weighted_balance_table

%matplotlib inline

# show effective-sample-size / trimming diagnostics logged during fit()
logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)

## 1. Synthetic data: two genuinely different, patient-level populations

Unlike the MAIC demo (which reads a target's moments off a published Table 1), IPTW needs
patient-level covariates on *both* sides so it can fit a pool-vs-target classifier. We use
`generate_toy_dataset`'s pool and target populations directly, with no aggregation step.

In [ ]:
m = generate_toy_dataset(n_pool=2000, n_target=300, seed=11)
pool_df = m.get_population("pool").drop(columns=[m.population_col])
target_df = m.get_population("target").drop(columns=[m.population_col])

headers = MatchingHeaders(numeric=["age", "weight", "height"], categoric=["gender", "haircolor"])
matching_data = MatchingData(pool=pool_df, target=target_df, headers=headers)
matching_data.describe()

## 2. Fit `IPTWWeighter`

As with the other weighters, `match()` fits (if needed) and returns a `MatchingData` with every
pool patient retained, plus a new `sample_weight` column. `weighted_balance_table()` again compares
each feature's target moment to the unweighted and weighted pool moment -- but this time, don't
expect the "weighted" column to land exactly on target the way it did for MAIC.

In [ ]:
weighter = IPTWWeighter(matching_data, verbose=True)
matched = weighter.match()

weighted_balance_table(weighter)

In [ ]:
print(f"Effective sample size: {weighter.effective_sample_size():.1f} / {len(pool_df)} pool patients")
print(f"Propensity model: {weighter.propensity_model}")
matched.get_population(matched.pool_name)["sample_weight"].describe()

## 3. Checking overlap: propensity score and weight distributions

IPTW relies on **positivity**: every pool patient needs a realistic chance of looking like the
target on the fitted model. A propensity score distribution piled up near 0 or 1, or a handful of
huge weights, is the practical warning sign -- the reweighting is leaning hard on a small,
possibly unrepresentative slice of the pool (the same concern `effective_sample_size()` summarizes
in one number).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

axes[0].hist(weighter.propensity_score, bins=30)
axes[0].set_title(r"Pool propensity scores, $\hat{p}(X) = P(\mathrm{target} \mid X)$")
axes[0].set_xlabel("propensity score")

axes[1].hist(weighter.weights, bins=30)
axes[1].set_title(r"Pool weights, $\hat{p} / (1 - \hat{p})$")
axes[1].set_xlabel("weight")

fig.tight_layout()

In [ ]:
plot_numeric_features(matched, weights="sample_weight", include_only=headers.numeric, col_wrap=3)
plt.suptitle("IPTW-weighted pool vs. target (numeric)", y=1.05);

In [ ]:
plot_categoric_features(matched, weights="sample_weight", include_only=headers.categoric, col_wrap=2)
plt.suptitle("IPTW-weighted pool vs. target (categoric)", y=1.05);

## 4. Trimming extreme weights

The main practical failure mode of IPTW is a small number of extreme weights, driven by pool
patients whose covariates make the model almost certain they're pool (weight -> 0) or almost
certain they're target (weight -> large). `trim_quantiles=(low, high)` clips fitted weights at the
given quantiles -- a standard bias/variance trade-off: a little exact balance is sacrificed in
exchange for a much more stable, lower-variance set of weights.

In [ ]:
trimmed = IPTWWeighter(matching_data, trim_quantiles=(0.05, 0.95), verbose=True)
trimmed.match()

print(f"Untrimmed: ESS={weighter.effective_sample_size():.1f}, max weight={weighter.weights.max():.2f}")
print(f"Trimmed:   ESS={trimmed.effective_sample_size():.1f}, max weight={trimmed.weights.max():.2f}, n_trimmed={trimmed.diagnostics['n_trimmed']}")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(weighter.weights, bins=30, alpha=0.6, label="untrimmed")
ax.hist(trimmed.weights, bins=30, alpha=0.6, label="trimmed (5th/95th pctile)")
ax.set_xlabel("weight")
ax.legend();

## 5. IPTW vs. MAIC on the same data

Both methods are solving the same underlying problem here -- reweight the pool onto the target --
so it's worth comparing them directly. `MAICWeighter` solves *exactly* for the target's mean (it
needs only the target's moments, not patient-level rows, though it works fine on patient-level
data too); `IPTWWeighter` only balances *approximately*, as a byproduct of a propensity model that
was never told to optimize balance directly.

In [ ]:
maic_weighter = MAICWeighter(matching_data, verbose=False)
maic_weighter.match()

iptw_table = weighted_balance_table(weighter).set_index("feature")["weighted_pool"]
maic_table = weighted_balance_table(maic_weighter).set_index("feature")["weighted_pool"]
target_table = weighted_balance_table(maic_weighter).set_index("feature")["target"]

comparison = pd.DataFrame({
    "target": target_table,
    "iptw_weighted_pool": iptw_table,
    "maic_weighted_pool": maic_table,
})
comparison["iptw_abs_residual"] = (comparison["iptw_weighted_pool"] - comparison["target"]).abs()
comparison["maic_abs_residual"] = (comparison["maic_weighted_pool"] - comparison["target"]).abs()
comparison

## Takeaways

- `IPTWWeighter` fits a pool-vs-target propensity model and weights the pool by $\hat p / (1 -
  \hat p)$; the target keeps weight 1 (the ATT estimand, target-as-reference), the same
  convention used by `MAICWeighter`/`EntropyBalanceWeighter`.
- It requires a **patient-level** target -- there's no aggregate-only ("published Table 1")
  variant, unlike MAIC.
- It does **not** solve for exact balance, so always check `weighted_balance_table()` and
  `effective_sample_size()` on the fitted result, the same as for any other weighter.
- `trim_quantiles` caps extreme weights, trading a little bias for a large reduction in variance --
  worth reaching for whenever a handful of patients end up dominating the effective sample size.
- Prefer `MAICWeighter`/`EntropyBalanceWeighter` when the target is aggregate-only or you need
  guaranteed exact balance; reach for `IPTWWeighter` when both populations are patient-level and
  you want the standard, most widely reported approach in the literature.